In [1]:
import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

/Users/juliasbardelatti/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df_junto = pd.read_csv("dados_paises_sus_estrangeiros.csv")

In [ ]:
#df_junto = df_junto.query("pais != 'RESERVADO'")

In [30]:
url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
resp = requests.get(url).json()

ibge = pd.DataFrame([
    {
        "codigo_ibge_7": str(item["id"]),       
        "codigo_ibge_6": str(item["id"])[:-1],  
        "nome_municipio": item["nome"]
    }
    for item in resp
])

df_junto['MUNIC_MOV'] = df_junto['MUNIC_MOV'].astype(str)

df_junto = df_junto.merge(
    ibge,
    left_on='MUNIC_MOV',
    right_on='codigo_ibge_6',
    how='left'
).drop(columns=['codigo_ibge_6'])

df_junto['codigo_ibge_7'] = df_junto['codigo_ibge_7'].astype(int)



In [34]:
df_municipios = (
    df_junto.groupby(["codigo_ibge_7", "nome_municipio"])
      .size()
      .reset_index(name="qtd_internacoes")
)

df_municipios.sort_values("qtd_internacoes", ascending=False).head(20)

,codigo_ibge_7,nome_municipio,qtd_internacoes
92,1400100,Boa Vista,32966
1467,3550308,São Paulo,31571
1545,4108304,Foz do Iguaçu,7298
1677,4204202,Chapecó,4883
1838,4314902,Porto Alegre,4445
1686,4205407,Florianópolis,3980
1709,4209102,Joinville,3588
1539,4106902,Curitiba,3563
71,1302603,Manaus,3265
1524,4104808,Cascavel,3093


In [35]:
df_pop = pd.read_excel("populacao_municipio_2022.xlsx")
df_pop = df_pop[df_pop['cod_mun'].str.isnumeric()]   # remove linha final
df_pop['cod_mun'] = df_pop['cod_mun'].astype(int)


In [36]:
df_merged = df_municipios.merge(
    df_pop,
    left_on="codigo_ibge_7",
    right_on="cod_mun",
    how="left"
)


In [40]:
df_merged['taxa_internacoes'] = (df_merged['qtd_internacoes'] / df_merged['pop']) * 100000
df_merged.sort_values("taxa_internacoes", ascending=False).head(20).to_clipboard(decimal=",")

